# ONIA · 2.6 Modele ansamblu

Suport de lucru: voting, bagging, Random Forest, Gradient Boosting, AdaBoost și stacking. Exemplele folosesc doar biblioteci Python uzuale și scikit-learn.

Pentru un experiment reproductibil, alegem explicit `random_state`; pentru comparații corecte păstrăm datele de validare separate.


## Vot și medie de la zero



In [ ]:
predictii_clasificare = [1, 0, 1, 1, 0]
voturi_pentru_1 = sum(predictii_clasificare)
rezultat = int(voturi_pentru_1 > len(predictii_clasificare) / 2)
print(rezultat)

predictii_regresie = [17, 22, 18]
print(sum(predictii_regresie) / len(predictii_regresie))


## Bootstrap cu liste Python



In [ ]:
import random

random.seed(7)
exemple = ['E1', 'E2', 'E3', 'E4', 'E5']
bootstrap = []
for _ in range(len(exemple)):
    pozitie = random.randrange(len(exemple))
    bootstrap.append(exemple[pozitie])
print(bootstrap)


## Bagging simplificat: învățare de praguri

Modelul de bază este un arbore cu o singură împărțire (decision stump). Este o implementare didactică simplificată: nu reproduce toate detaliile clasei <code>BaggingClassifier</code>.

In [ ]:
import random

def invata_prag(exemple):
    # exemple = lista de perechi (x, eticheta 0/1)
    valori = sorted(set(x for x, y in exemple))
    praguri = [(a + b) / 2 for a, b in zip(valori, valori[1:])]
    cel_mai_bun = (float('inf'), 0.0, 0, 0)
    for prag in praguri:
        st = [y for x, y in exemple if x <= prag]
        dr = [y for x, y in exemple if x > prag]
        et_st = int(sum(st) * 2 > len(st))
        et_dr = int(sum(dr) * 2 > len(dr))
        erori = sum(y != (et_st if x <= prag else et_dr)
                    for x, y in exemple)
        if erori < cel_mai_bun[0]:
            cel_mai_bun = (erori, prag, et_st, et_dr)
    return cel_mai_bun[1:]

exemple = [(1, 0), (2, 0), (3, 1), (4, 1), (5, 1)]
random.seed(2)
modele = []
for _ in range(7):
    esantion = random.choices(exemple, k=len(exemple))
    modele.append(invata_prag(esantion))

def predictie(model, x):
    prag, st, dr = model
    return st if x <= prag else dr

voturi = [predictie(model, 3.2) for model in modele]
print(voturi, int(sum(voturi) > len(voturi) / 2))


## Random Forest pentru clasificare



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model = RandomForestClassifier(
    n_estimators=100, max_features='sqrt',
    min_samples_leaf=2, random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, pred):.3f}')
print('Probabilități:', model.predict_proba(X_test[:1]))


## RandomForestRegressor



In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X, y = make_regression(n_samples=250, n_features=5,
                       noise=16, random_state=3)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.25, random_state=3
)
model = RandomForestRegressor(n_estimators=120,
                              min_samples_leaf=3,
                              random_state=3)
model.fit(X_train, y_train)
print('MAE:', round(mean_absolute_error(y_test, model.predict(X_test)), 2))


## Boosting from scratch cu arbori de un nivel

<strong>Ce este „from scratch” aici?</strong> Am implementat noi algoritmul de boosting — predictorul inițial, calculul reziduurilor și actualizarea predicțiilor. Pentru arborii simpli folosim o clasă scikit-learn, astfel încât ideea principală să fie clară înaintea unei implementări complete CART.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
import numpy as np

X = np.array([[1], [2], [3], [4], [5], [6]], dtype=float)
y = np.array([2, 3, 3, 7, 8, 9], dtype=float)
learning_rate = 0.3
predictie = np.full(len(y), y.mean())
arbori = []

for etapa in range(5):
    reziduuri = y - predictie
    arbore = DecisionTreeRegressor(max_depth=1, random_state=etapa)
    arbore.fit(X, reziduuri)
    corectie = arbore.predict(X)
    predictie += learning_rate * corectie
    arbori.append(arbore)
    print(etapa + 1, 'MSE:', round(np.mean((y - predictie)**2), 4))


### Boosting pur Python: toate etapele explicite

Această variantă folosește numai liste și bucle Python; implementează și alegerea pragului pentru fiecare arbore cu un singur nod.

In [ ]:

# Gradient Boosting pentru regresie, fără biblioteci externe
x = [1, 2, 3, 4, 5, 6]
y = [2, 3, 3, 7, 8, 9]
predictie = [sum(y) / len(y)] * len(y)
eta = 0.3

for etapa in range(4):
    reziduuri = [y[i] - predictie[i] for i in range(len(y))]
    best = None
    # Un arbore cu un singur nod de decizie (stump)
    for poz in range(1, len(x)):
        st = reziduuri[:poz]
        dr = reziduuri[poz:]
        medie_st = sum(st) / len(st)
        medie_dr = sum(dr) / len(dr)
        eroare = (sum((v - medie_st)**2 for v in st) +
                  sum((v - medie_dr)**2 for v in dr))
        if best is None or eroare < best[0]:
            best = (eroare, poz, medie_st, medie_dr)
    _, poz, medie_st, medie_dr = best
    for i in range(len(y)):
        corectie = medie_st if i < poz else medie_dr
        predictie[i] += eta * corectie
    mse = sum((y[i] - predictie[i])**2 for i in range(len(y))) / len(y)
    print(f'Etapa {etapa + 1}: MSE = {mse:.4f}')


## AdaBoostClassifier cu arbori simpli



In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=240, n_features=6,
                           n_informative=4, n_redundant=0,
                           random_state=5)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.25, random_state=5, stratify=y
)
model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=5),
    n_estimators=40, random_state=5
)
model.fit(X_train, y_train)
print('Accuracy:', round(model.score(X_test, y_test), 3))


## GradientBoostingRegressor cu validare



In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

X, y = make_regression(n_samples=280, n_features=5,
                       noise=23, random_state=8)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.25, random_state=8
)
model = GradientBoostingRegressor(
    n_estimators=80, learning_rate=.05,
    max_depth=2, random_state=8
)
model.fit(X_train, y_train)
print('MSE:', round(mean_squared_error(y_test,
                                       model.predict(X_test)), 2))


## HistGradientBoostingClassifier



In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=4, stratify=y, test_size=.25
)
model = HistGradientBoostingClassifier(
    max_iter=80, learning_rate=.07,
    max_leaf_nodes=12, l2_regularization=1.0,
    random_state=4
)
model.fit(X_train, y_train)
print('Accuracy:', round(model.score(X_test, y_test), 3))


## VotingClassifier cu pipeline pentru K-NN



In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_iris(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, stratify=y, test_size=.25, random_state=9
)
model = VotingClassifier(
    estimators=[
        ('log', make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))),
        ('knn', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))),
        ('arbore', DecisionTreeClassifier(max_depth=4, random_state=9)),
    ], voting='soft', weights=[2, 1, 1]
)
model.fit(Xtr, ytr)
print('Accuracy:', round(model.score(Xte, yte), 3))


## StackingClassifier



In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_iris(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=.25, stratify=y, random_state=11
)
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=11)
model = StackingClassifier(
    estimators=[
        ('forest', RandomForestClassifier(n_estimators=50, random_state=11)),
        ('knn', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)))
    ],
    final_estimator=LogisticRegression(max_iter=500),
    cv=cv
)
model.fit(Xtr, ytr)
print('Accuracy:', round(model.score(Xte, yte), 3))


## Coloane numerice și categorice



In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

date = pd.DataFrame({
    'varsta': [14, 16, 15, 17, 14, 18],
    'dispozitiv': ['mobil', 'pc', 'pc', 'tableta', 'mobil', 'pc'],
    'rezultat': [0, 1, 1, 0, 0, 1]
})
X = date[['varsta', 'dispozitiv']]
y = date['rezultat']
preprocesare = ColumnTransformer([
    ('categorice', OneHotEncoder(handle_unknown='ignore'), ['dispozitiv'])
], remainder='passthrough')
model = Pipeline([
    ('preprocesare', preprocesare),
    ('classifier', RandomForestClassifier(n_estimators=20, random_state=0))
])
model.fit(X, y)
print(model.predict(pd.DataFrame({'varsta':[16], 'dispozitiv':['mobil']})))


## Arbore, Random Forest și Boosting comparate corect

Aceleiași împărțiri trebuie aplicate tuturor modelelor; altfel compari scoruri influențate de seturi de test diferite.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

X, y = make_moons(n_samples=320, noise=.23, random_state=14)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=.30, stratify=y, random_state=14
)
modele = {
    'Arbore': DecisionTreeClassifier(max_depth=5, random_state=14),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=14),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=70,
                                                    learning_rate=.07,
                                                    max_depth=2, random_state=14)
}
for nume, model in modele.items():
    model.fit(Xtr, ytr)
    rezultat = accuracy_score(yte, model.predict(Xte))
    print(f'{nume}: {rezultat:.3f}')


## Exerciții pentru studiu individual

1. Schimbă `n_estimators` în Random Forest și compară scorul pe validare.
2. Compară un singur arbore cu o pădure, păstrând exact aceeași împărțire a datelor.
3. Desenează evoluția MSE după fiecare etapă de Gradient Boosting.
4. Arată manual cum un exemplu poate lipsi dintr-un eșantion bootstrap.
5. Explică de ce meta-modelul din stacking trebuie antrenat pe predicții out-of-fold.
6. Demonstrează cu un contraexemplu că votarea a mai multor modele nu îmbunătățește garantat rezultatul.
